In [1]:
!pip install PyPDF2

In [2]:
import re
import requests
import nltk
import numpy as np
from PyPDF2 import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt")

[nltk_data] Downloading package punkt to /Users/anirudh/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
!wget 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7vgNfis17dQfjHAiIKkBOg/The-Daily-Dish-FAQ.pdf'
faq_pdf_path = "The-Daily-Dish-FAQ.pdf"

--2026-06-02 10:06:18--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7vgNfis17dQfjHAiIKkBOg/The-Daily-Dish-FAQ.pdf
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.45.118.108
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.45.118.108|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 53993 (53K) [application/pdf]
Saving to: ‘The-Daily-Dish-FAQ.pdf’

The-Daily-Dish-FAQ. 100%[===================>]  52.73K  --.-KB/s    in 0.1s    

2026-06-02 10:06:22 (398 KB/s) - ‘The-Daily-Dish-FAQ.pdf’ saved [53993/53993]



In [4]:
class MemoryAgent:
    def __init__(self):
        # Initialize an empty dictionary to store agent memory
        # This acts as shared context across multiple agents
        self.memory = {}

    def store(self, key, value):
        """
        Store information in memory using a key-value pair.
        Example:
        key   -> 'last_weather_query'
        value -> 'Weather in Bangalore is 28°C'
        """
        self.memory[key] = value

    def recall(self, key=None):
        """
        Retrieve information from memory.
        
        - If a specific key is provided, return the stored value for that key.
        - If no key is provided, return the entire memory dictionary.
        
        This allows agents to:
        - Maintain conversation context
        - Avoid repeating work
        - Adapt responses based on past interactions
        """
        if key:
            return self.memory.get(key)
        return self.memory


In [5]:
class WeatherAgent:
    # Initialize the Weather Agent
    # - api_key: OpenWeather API key
    # - memory_agent: shared memory agent to store past weather data
    def __init__(self, api_key, memory_agent):
        self.api_key = api_key              # Store the API key
        self.memory = memory_agent          # Reference to the memory agent
        self.url = "http://api.openweathermap.org/data/2.5/weather"  # Weather API endpoint

    # Generate a weather response for a given city
    def answer(self, city):
        # Prepare query parameters for the API call
        params = {
            "q": city,                      # City name
            "appid": self.api_key,          # API authentication key
            "units": "metric"               # Return temperature in Celsius
        }

        # Call the OpenWeather API
        res = requests.get(self.url, params=params)

        # Handle API errors or failed requests
        if res.status_code != 200:
            return "I couldn’t retrieve the weather right now."

        # Parse the JSON response
        data = res.json()

        # Retrieve previous weather data for this city from memory (if available)
        previous = self.memory.recall(city)

        # Store the current weather data in memory for future context
        self.memory.store(city, data["main"])

        # Construct a natural-language weather response
        response = (
            f"The current weather in {city} is {data['weather'][0]['description']} "
            f"with a temperature of {data['main']['temp']}°C."
        )

        # If previous data exists, add contextual comparison
        if previous:
            response += f" Earlier it was {previous['temp']}°C."

        # Return the final response to the user
        return response


In [6]:
# Path to the Daily Dish FAQ PDF file
FAQ_PDF = "The-Daily-Dish-FAQ.pdf"
def load_faq_pdf(pdf_path):
    # Create a PDF reader object
    reader = PdfReader(pdf_path)
    
    # Initialize an empty string to store extracted text
    text = ""
    
    # Loop through each page in the PDF
    for page in reader.pages:
        # Extract text from the page and append it
        text += page.extract_text()
    
    # Return the complete extracted text
    return text

# Extract all text content from the FAQ PDF
faq_text = load_faq_pdf(FAQ_PDF)


In [7]:
def clean_text(text):
    # Replace multiple spaces, newlines, or tabs with a single space
    cleaned_text = re.sub(r"\s+", " ", text)
    
    # Remove leading and trailing whitespace
    return cleaned_text.strip()

In [8]:
# Function to parse a FAQ text into structured question-answer pairs
def parse_faq(text):
    # Initialize an empty list to store the parsed Q&A pairs
    faq_pairs = []

    # Define a regex pattern to match "Q: <question> A: <answer>" blocks
    # - (.*?) captures the question text lazily
    # - (.*?)(?=\n\s*\d+\.\s*Q:|\Z) captures the answer lazily until the next numbered question or end of text
    pattern = r"Q:\s*(.*?)\s*A:\s*(.*?)(?=\n\s*\d+\.\s*Q:|\Z)"

    # Find all matches of the pattern in the text
    # re.DOTALL allows '.' to match newline characters so multi-line answers are captured
    matches = re.findall(pattern, text, re.DOTALL)

    # Loop over each matched question-answer tuple
    for q, a in matches:
        # Append a dictionary with cleaned question and answer to the list
        # - clean_text() removes extra spaces and normalizes the text
        # - question is converted to lowercase for consistent indexing/search
        faq_pairs.append({
            "question": clean_text(q.lower()),
            "answer": clean_text(a)
        })
    
    # Return the list of structured Q&A dictionaries
    return faq_pairs

# Parse the FAQ text into structured data
faq_data = parse_faq(faq_text)

# Extract just the list of questions from the structured data
faq_questions = [item["question"] for item in faq_data]

# Extract just the list of answers from the structured data
faq_answers = [item["answer"] for item in faq_data]


In [9]:
# Class to preprocess and enhance user queries
class QueryProcessor:
    def process(self, query):
        # Convert the entire query to lowercase for consistency
        query = query.lower()

        # Remove all punctuation and special characters, keeping only letters, numbers, and spaces
        query = re.sub(r"[^\w\s]", "", query)

        # Define a dictionary of synonyms to expand the query
        # Keys are words to look for in the query
        # Values are additional terms to append if the key is found
        synonyms = {
            "location": "located address",
            "where": "located address",
            "reservation": "reserve booking",
            "menu": "food dishes",
            "fish": "seafood"
        }

        # Loop through the synonyms dictionary
        for k, v in synonyms.items():
            # If the keyword exists in the query, append its corresponding synonym
            # This helps improve matching or search results
            if k in query:
                query += " " + v

        # Return the processed and enhanced query
        return query


In [10]:
# Agent class for handling FAQ-style question-answer interactions
class DailyDishAgent:
    def __init__(self, questions, answers):
        # Store the list of answers corresponding to the FAQ questions
        self.answers = answers

        # Initialize a TF-IDF vectorizer for converting text into numerical vectors
        # - stop_words="english" removes common English words (like "the", "is") for better focus on keywords
        # - ngram_range=(1, 2) considers both single words (unigrams) and pairs of words (bigrams) for richer text representation
        self.vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2)
        )

        # Fit the vectorizer to the questions and transform them into TF-IDF vectors
        # This creates a matrix where each row represents a question and columns represent weighted terms
        self.doc_vectors = self.vectorizer.fit_transform(questions)

      # Method to find the most relevant answer for a given user query
    def answer(self, query):
        # Transform the user query into a TF-IDF vector using the same vectorizer as the questions
        query_vector = self.vectorizer.transform([query])
    
        # Compute cosine similarity between the query vector and all stored question vectors
        # - cosine similarity measures how close the query is to each question
        # - similarities[0] gives a 1D array of similarity scores
        similarities = cosine_similarity(query_vector, self.doc_vectors)[0]
    
        # Find the index of the question with the highest similarity score
        best_idx = np.argmax(similarities)
    
        # If the highest similarity is below a threshold (0.08), consider it "no match"
        # - This prevents returning irrelevant answers
        if similarities[best_idx] < 0.08:
            return None
    
        # Return the answer corresponding to the most similar question
        return self.answers[best_idx]



In [11]:
# Function to route a user query to the appropriate agent
def route_query(query):
    # Define keywords that indicate the query is related to weather
    weather_keywords = [
        "weather", "rain", "raining", "forecast",
        "temperature", "hot", "cold", "humidity"
    ]

    # Convert the query to lowercase for case-insensitive matching
    query = query.lower()

    # Check if any weather-related keyword exists in the query
    for word in weather_keywords:
        if word in query:
            # Route to the weather agent
            return "weather"

    # If no weather keywords are found, route to the DailyDish agent
    return "daily_dish"


In [ ]:
# Initialize the query processor to clean and expand user queries
query_processor = QueryProcessor()

# Initialize a memory agent to store and recall context if needed
memory_agent = MemoryAgent()

# 👉 Replace with your real API key for the weather API
WEATHER_API_KEY = "enter your api key"

# Initialize the WeatherAgent with API key and memory
weather_agent = WeatherAgent(WEATHER_API_KEY, memory_agent)

# Initialize the DailyDishAgent with FAQ questions and answers
daily_dish_agent = DailyDishAgent(faq_questions, faq_answers)

# Define the city for weather queries
RESTAURANT_CITY = "New york"


In [15]:
def chatbot(user_question):
    # Determine which agent should handle the query (Weather or DailyDish)
    route = route_query(user_question)

    # Preprocess the query (lowercase, clean text, expand synonyms)
    processed = query_processor.process(user_question)

    # If the query is weather-related, get answer from WeatherAgent
    if route == "weather":
        return weather_agent.answer(RESTAURANT_CITY)

    # Otherwise, get answer from DailyDishAgent
    answer = daily_dish_agent.answer(processed)
    if answer:
        return answer

    # Fallback message for unrecognized queries
    return "I’m not sure about that. Please ask a question related to The Daily Dish."


In [16]:
print("🍽️ Welcome to The Daily Dish Chatbot!")
print("Type 'exit' to end the conversation.\n")
while True:
    user_input = input("You: ")

    # Exit conditions for ending the chat
    if user_input.lower() in ["exit", "quit", "bye"]:
        print("Chatbot: 👋 Thanks for visiting The Daily Dish!")
        break

    # Process the query and print the chatbot's response
    print("Chatbot:", chatbot(user_input))
    print()



🍽️ Welcome to The Daily Dish Chatbot!
Type 'exit' to end the conversation.



You:  Give me a good recipie


Chatbot: I’m not sure about that. Please ask a question related to The Daily Dish.



You:  give me apple juice recipie


Chatbot: I’m not sure about that. Please ask a question related to The Daily Dish.



You:  What is th weather in Seattle


Chatbot: I couldn’t retrieve the weather right now.



You:  exit


Chatbot: 👋 Thanks for visiting The Daily Dish!
